<a href="https://colab.research.google.com/github/zxn-999/BSE_CNN/blob/main/Unet_3_21_dice%2Bweight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install segmentation-models-pytorch --quiet
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from sklearn.metrics import confusion_matrix
from matplotlib.colors import ListedColormap

# 自定义颜色（4个phase）
colors = [
    (0, 0, 0),        # Class 0 - 黑色 (孔隙)
    (1, 0, 0),        # Class 1 - 红色
    (0, 1, 0),        # Class 2 - 绿色
    (1, 1, 0)         # Class 3 - 蓝色
]

cmap = ListedColormap(colors)

def save_tif_gray(path, img):
    if img.dtype != np.uint8:
        img = (img * 255).astype(np.uint8)
    cv2.imwrite(path, img)

def save_tif_color(path, label_map, cmap):
    colored = cmap(label_map)[:, :, :3]
    colored = (colored * 255).astype(np.uint8)
    colored = cv2.cvtColor(colored, cv2.COLOR_RGB2BGR)
    cv2.imwrite(path, colored)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.7 MB/s eta 0:00:00


In [3]:
# 1. 基础配置与路径
# ==========================================
IMAGE_DIR = "/content/drive/MyDrive/BSE/images"
MASK_DIR = "/content/drive/MyDrive/BSE/mask"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
EPOCHS = 10
PATCH_SIZE = 256
NUM_CLASSES = 4

# 创建保存结果的文件夹
OUTPUT_DIR = "/content/drive/MyDrive/BSE/output_Unet"

VAL_IMG_DIR = os.path.join(OUTPUT_DIR, "validate/images")
VAL_GT_DIR  = os.path.join(OUTPUT_DIR, "validate/gt")
VAL_PRED_DIR= os.path.join(OUTPUT_DIR, "validate/pred")

TEST_IMG_DIR = os.path.join(OUTPUT_DIR, "test/images")
TEST_GT_DIR  = os.path.join(OUTPUT_DIR, "test/gt")
TEST_PRED_DIR= os.path.join(OUTPUT_DIR, "test/pred")

In [4]:
# 2. 数据集类定义
# ==========================================
class BSEPatchDataset(Dataset):
    def __init__(self, image_dir, mask_dir, patch_size=256):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.patch_size = patch_size
        self.image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(".tif")])
        self.patches = []
        self._prepare_patches()

    def _prepare_patches(self):
        for img_name in self.image_files:
            img_path = os.path.join(self.image_dir, img_name)
            mask_path = os.path.join(self.mask_dir, img_name.replace(".tif", "_mask.tif"))
            if not os.path.exists(mask_path): continue

            image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            h, w = image.shape
            for y in range(0, h - self.patch_size + 1, self.patch_size):
                for x in range(0, w - self.patch_size + 1, self.patch_size):
                    self.patches.append((img_path, mask_path, x, y))

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        img_path, mask_path, x, y = self.patches[idx]
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        image = image[y:y+self.patch_size, x:x+self.patch_size]
        mask = mask[y:y+self.patch_size, x:x+self.patch_size]

        image = torch.from_numpy(image).float().unsqueeze(0) / 255.0
        mask = torch.from_numpy(mask).long()
        mask = torch.clamp(mask, min=0, max=3)
        return image, mask

In [5]:
# 3. 评价指标计算函数
# ==========================================
def calculate_metrics(pred, target, num_classes=4):
    """计算准确率、精确率、召回率、假阳性率"""
    pred = pred.view(-1).cpu().numpy()
    target = target.view(-1).cpu().numpy()
    cm = confusion_matrix(target, pred, labels=range(num_classes))

    results = []
    for i in range(num_classes):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - tp - fn - fp

        accuracy = (tp + tn) / np.sum(cm) if np.sum(cm) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        fall_out = fp / (fp + tn) if (fp + tn) > 0 else 0
    # 增加IoU和F1score
        IoU = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
        F1 = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
        results.append([accuracy, precision, recall, fall_out, IoU, F1])
    return np.array(results)

In [6]:
# 4. 数据划分与加载
# ==========================================
full_dataset = BSEPatchDataset(IMAGE_DIR, MASK_DIR, PATCH_SIZE)
total_size = len(full_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_ds, val_ds, test_ds = random_split(full_dataset, [train_size, val_size, test_size],
                                         generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)

print ("Total patches:", total_size)

Total patches: 8638


In [7]:
# 4.5 计算类别权重（基于数据分布）
# ==========================================
print("Calculating class weights...")

class_counts = np.zeros(NUM_CLASSES)

for i in range(len(train_ds)):
    _, mask = train_ds[i]
    m = mask.numpy()
    for c in range(NUM_CLASSES):
        class_counts[c] += np.sum(m == c)

class_freq = class_counts / np.sum(class_counts)

# 使用 sqrt 平滑（推荐）
weights = 1 / np.sqrt(class_freq + 1e-6)
weights = weights / weights.min()

print("Class frequencies:", class_freq)
print("Class weights:", weights)

weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)

Calculating class weights...
Class frequencies: [0.10926089 0.3438717  0.42917157 0.11769583]
Class weights: [1.98189946 1.1171645  1.         1.90956124]


In [8]:
# 5. 模型、损失函数与优化器
# ==========================================
model = smp.Unet(encoder_name="resnet34", in_channels=1, classes=NUM_CLASSES).to(DEVICE)
dice_loss = smp.losses.DiceLoss(mode='multiclass')
ce_loss = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [11]:
# 6. 训练与验证循环
# ==========================================
for epoch in range(EPOCHS):
    # --- 训练阶段 ---
    model.train()
    train_loss = 0
    running_loss_d = 0
    running_loss_ce = 0
    for images, masks in train_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss_d = dice_loss(outputs, masks)
        loss_ce = ce_loss(outputs, masks)
        loss = loss_d + loss_ce
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        running_loss_d += loss_d.item() * images.size(0)
        running_loss_ce += loss_ce.item() * images.size(0)

    # --- 验证阶段 ---
    model.eval()
    val_metrics = np.zeros((NUM_CLASSES, 6))
    with torch.no_grad():
        for i, (images, masks) in enumerate(val_loader):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            val_metrics += calculate_metrics(preds, masks)
            # 每个 Epoch 保存前 3 个验证结果图
            if i < 3:
                img_np = images[0,0].cpu().numpy()
                pred_np = preds[0].cpu().numpy()
                mask_np = masks[0].cpu().numpy()

                save_tif_gray(
                    os.path.join(VAL_IMG_DIR, f"epoch{epoch}_sample{i}.tif"),
                    img_np
                )

                save_tif_color(
                    os.path.join(VAL_GT_DIR, f"epoch{epoch}_sample{i}.tif"),
                    mask_np, cmap
                )

                save_tif_color(
                    os.path.join(VAL_PRED_DIR, f"epoch{epoch}_sample{i}.tif"),
                    pred_np, cmap
                )
    avg_val_metrics = val_metrics / len(val_loader)
    epoch_loss_d = running_loss_d / len(train_loader)
    epoch_loss_ce = running_loss_ce / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] "
      f"Total: {train_loss/len(train_loader):.4f} | "
      f"Dice: {epoch_loss_d:.4f} | "
      f"CE: {epoch_loss_ce:.4f}")
    print(f"\n--- Epoch [{epoch+1}/{EPOCHS}] Validation Summary ---")
    for c in range(NUM_CLASSES):
        print(f"Class {c}: Recall={avg_val_metrics[c, 2]:.4f}, Fall-out={avg_val_metrics[c, 3]:.4f}, IoU={avg_val_metrics[c,4]:.4f}, F1={avg_val_metrics[c,5]:.4f}")
    print("-" * 30)

Epoch [1/10] Total: 5.9597 | Dice: 2.3145 | CE: 3.6452

--- Epoch [1/10] Validation Summary ---
Class 0: Recall=0.9338, Fall-out=0.0391, IoU=0.7115, F1=0.8127
Class 1: Recall=0.8517, Fall-out=0.0850, IoU=0.7393, F1=0.8419
Class 2: Recall=0.8068, Fall-out=0.0466, IoU=0.7618, F1=0.8595
Class 3: Recall=0.9110, Fall-out=0.0405, IoU=0.7041, F1=0.8105
------------------------------
Epoch [2/10] Total: 4.5215 | Dice: 1.6807 | CE: 2.8408

--- Epoch [2/10] Validation Summary ---
Class 0: Recall=0.8619, Fall-out=0.0168, IoU=0.7505, F1=0.8430
Class 1: Recall=0.9036, Fall-out=0.0693, IoU=0.7984, F1=0.8846
Class 2: Recall=0.8870, Fall-out=0.0601, IoU=0.8203, F1=0.8975
Class 3: Recall=0.8739, Fall-out=0.0167, IoU=0.7523, F1=0.8477
------------------------------
Epoch [3/10] Total: 4.1167 | Dice: 1.5347 | CE: 2.5820

--- Epoch [3/10] Validation Summary ---
Class 0: Recall=0.9027, Fall-out=0.0237, IoU=0.7574, F1=0.8460
Class 1: Recall=0.8946, Fall-out=0.0646, IoU=0.7962, F1=0.8828
Class 2: Recall=0.88

In [12]:
# 7. 最终测试阶段
# ==========================================
print("\n--- Starting Final Test ---")
import cv2
import os
model.eval()
# 用于累加所有 Patch 的指标
test_metrics_accumulator = np.zeros((NUM_CLASSES, 6)) # [Accuracy, Precision, Recall, Fall-out, IoU, F1]
with torch.no_grad():
    for i, (images, masks) in enumerate(test_loader):
        img_np = images[0,0].cpu().numpy()
        pred_np = preds[0].cpu().numpy()
        mask_np = masks[0].cpu().numpy()

        # ===== 原图 =====
        save_tif_gray(
            os.path.join(TEST_IMG_DIR, f"test_{i}.tif"),
            img_np
        )

        # ===== GT =====
        save_tif_color(
            os.path.join(TEST_GT_DIR, f"test_{i}.tif"),
            mask_np, cmap
        )

        # ===== Prediction =====
        save_tif_color(
            os.path.join(TEST_PRED_DIR, f"test_{i}.tif"),
            pred_np, cmap
        )

        # 计算当前 Patch 的指标并累加
        patch_metrics = calculate_metrics(preds, masks, num_classes=NUM_CLASSES)
        test_metrics_accumulator += patch_metrics

        # 保存所有测试集的预测结果图
        plt.savefig(os.path.join(OUTPUT_DIR, "test", f"test_{i}_color.tif"),
            bbox_inches='tight', pad_inches=0)

# 计算所有测试样本的平均值
final_metrics = test_metrics_accumulator / len(test_loader)
# --- 打印每个 Class 的单独指标 ---
print(f"{'Class':<10} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10} | {'Fall-out':<10} | {'IoU':<10} | {'F1':<10}")
print("-" * 65)
for c in range(NUM_CLASSES):
    print(f"Class {c:<6} | {final_metrics[c,0]:.4f}   | {final_metrics[c,1]:.4f}    | {final_metrics[c,2]:.4f} | {final_metrics[c,3]:.4f} | {final_metrics[c,4]:.4f} | {final_metrics[c,5]:.4f}")

# --- 计算并打印总指标 (Macro Average) ---
# 总指标即所有类别指标的平均值
total_accuracy  = np.mean(final_metrics[:, 0])
total_precision = np.mean(final_metrics[:, 1])
total_recall    = np.mean(final_metrics[:, 2])
total_fall_out  = np.mean(final_metrics[:, 3])
total_IoU       = np.mean(final_metrics[:, 4])
total_F1        = np.mean(final_metrics[:, 5])

print("-" * 65)
print(f"{'Average':<10} | {total_accuracy:.4f}   | {total_precision:.4f}    | {total_recall:.4f} | {total_fall_out:.4f}  | {total_IoU:.4f}  | {total_F1:.4f}")
print("="*50)


--- Starting Final Test ---
Class      | Accuracy   | Precision  | Recall     | Fall-out   | IoU        | F1        
-----------------------------------------------------------------
Class 0      | 0.8860   | 0.1125    | 0.0062 | 0.0048 | 0.0055 | 0.0106
Class 1      | 0.6246   | 0.3460    | 0.1021 | 0.0996 | 0.0832 | 0.1528
Class 2      | 0.4868   | 0.4321    | 0.6103 | 0.6089 | 0.3362 | 0.4967
Class 3      | 0.6688   | 0.1194    | 0.2867 | 0.2822 | 0.0842 | 0.1442
-----------------------------------------------------------------
Average    | 0.6665   | 0.2525    | 0.2513 | 0.2489  | 0.1273  | 0.2011


<Figure size 640x480 with 0 Axes>

In [ ]:
print("\n" + "="*50 + "\n最终测试结果报告\n" + "="*50)
model.eval()
test_metrics_acc = np.zeros((NUM_CLASSES, 4))

with torch.no_grad():
    for i, (images, masks) in enumerate(test_loader):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        preds = torch.argmax(model(images), dim=1)
        test_metrics_acc += calculate_metrics(preds, masks)

        # 保存彩色的预测结果图 (不含坐标轴)
        plt.figure(figsize=(5,5))
        plt.imshow(preds[0].cpu().numpy(), cmap='jet')
        plt.axis('off')
        plt.savefig(f"output_Unet/test/test_{i}_color.tif", bbox_inches='tight', pad_inches=0)
        plt.close()

# 打印结果表
final_test_m = test_metrics_acc / len(test_loader)
print(f"{'类别':<8} | {'Accuracy':<8} | {'Precision':<8} | {'Recall':<8} | {'Fall-out':<8}")
print("-" * 60)
for c in range(NUM_CLASSES):
    print(f"Class {c:<4} | {final_test_m[c,0]:.4f}   | {final_test_m[c,1]:.4f}    | {final_test_m[c,2]:.4f}   | {final_test_m[c,3]:.4f}")

# 打印总平均
m_avg = np.mean(final_test_m, axis=0)
print("-" * 60)
print(f"总平均   | {m_avg[0]:.4f}   | {m_avg[1]:.4f}    | {m_avg[2]:.4f}   | {m_avg[3]:.4f}")
print("="*50)


最终测试结果报告
类别       | Accuracy | Precision | Recall   | Fall-out
------------------------------------------------------------
Class 0    | 0.9809   | 0.8811    | 0.9154   | 0.0143
Class 1    | 0.9419   | 0.9017    | 0.9292   | 0.0550
Class 2    | 0.9399   | 0.9404    | 0.9078   | 0.0471
Class 3    | 0.9789   | 0.9040    | 0.8607   | 0.0116
------------------------------------------------------------
总平均   | 0.9604   | 0.9068    | 0.9033   | 0.0320
